# 00 — FoodVision AI | Dataset Audit, EDA & Split Verification

**Purpose:** One place to inspect the original Food-270 images, understand the cleaned dataset and class balance, verify the established 70/15/15 splits, and migrate the small, reproducible *split metadata* out of `ml/reports/`.

**Important:** The original dataset-cleaning decisions and split-generation logic lived in separate scripts that were not included in the two supplied notebooks. This notebook audits the existing outputs; it **does not claim to reproduce the original cleaning decisions** or regenerate the exact splits. Keep the original scripts and manifests until verified. It never retrains a model or edits the original dataset.

## 1. Configure paths

In [ ]:
from pathlib import Path
from collections import Counter
import csv, json, random, shutil
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

DATASET_ROOT = Path(r"C:\deep_learning_projects\archive\images")
# Search from this notebook's working directory for the existing split files.
current = Path.cwd()
PROJECT_ROOT = next((p for p in (current, *current.parents)
                     if (p / 'ml/data/splits/train.csv').exists()
                     or (p / 'ml/data/splits/train.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open Jupyter from the foodvision-ai project root or ml/notebooks.')
OLD_SPLITS = PROJECT_ROOT / 'ml/data/splits'
NEW_SPLITS = PROJECT_ROOT / 'ml/data/splits'
SPLITS_DIR = NEW_SPLITS if (NEW_SPLITS / 'train.csv').exists() else OLD_SPLITS
print('Image folder:', DATASET_ROOT, 'exists:', DATASET_ROOT.is_dir())
print('Splits directory:', SPLITS_DIR)
assert DATASET_ROOT.is_dir(), 'Check DATASET_ROOT'

## 2. Read the existing cleaned dataset splits

In [ ]:
splits = {name: pd.read_csv(SPLITS_DIR / f'{name}.csv')
          for name in ('train', 'val', 'test')}
required = {'relative_path', 'class_name', 'class_index'}
for name, df in splits.items():
    assert required.issubset(df.columns), f'{name}: missing required columns'
    assert df[list(required)].notna().all().all(), f'{name}: null values'
    print(f'{name:5s}: {len(df):,} images, {df.class_name.nunique()} classes')
print('Total selected/clean images:', sum(map(len, splits.values())))

## 3. Class distribution (EDA)

In [ ]:
train_counts = splits['train'].class_name.value_counts()
print('Train examples per class:')
print(train_counts.describe().round(1).to_string())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train_counts.hist(bins=25, ax=axes[0])
axes[0].set(title='Training examples per class', xlabel='Images', ylabel='Classes')
train_counts.sort_values().head(15).plot.barh(ax=axes[1])
axes[1].set(title='15 smallest training classes', xlabel='Images')
plt.tight_layout(); plt.show()

## 4. Confirm mapping, leakage and split consistency

In [ ]:
with (SPLITS_DIR / 'class_to_index.json').open(encoding='utf-8') as f:
    class_to_index = json.load(f)
print('Mapped food classes:', len(class_to_index))
assert len(class_to_index) == 270, 'Unexpected number of classes'
for name, df in splits.items():
    expected = df.class_name.map(class_to_index)
    assert expected.notna().all(), f'{name}: class missing from mapping'
    assert (expected.astype(int) == df.class_index).all(), f'{name}: class index mismatch'
    assert df.relative_path.is_unique, f'{name}: duplicate relative paths'
paths = {k: set(v.relative_path.str.replace('\\', '/', regex=False)) for k, v in splits.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    common = paths[a] & paths[b]
    print(f'{a}/{b} shared file paths:', len(common))
    assert not common, f'Leakage by relative file path: {a}/{b}'
print('Checks passed (path-level only; near-duplicates require separate image-hash analysis).')

## 5. Check missing images (can scan all 225k paths)

In [ ]:
missing = []
for name, df in splits.items():
    for rel in df.relative_path:
        file = DATASET_ROOT / Path(str(rel).replace('\\', '/'))
        if not file.is_file():
            missing.append((name, str(rel)))
print('Missing image files:', len(missing))
print('Examples:', missing[:10])
assert not missing, 'Fix missing image paths before training'

## 6. Display reproducible random samples

In [ ]:
items = splits['train'].sample(n=12, random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(12, 10))
for ax, row in zip(axes.flat, items.itertuples(index=False)):
    p = DATASET_ROOT / Path(str(row.relative_path).replace('\\', '/'))
    try:
        with Image.open(p) as img:
            ax.imshow(ImageOps.exif_transpose(img).convert('RGB'))
        ax.set_title(str(row.class_name), fontsize=10)
    except Exception as exc:
        ax.text(.05, .5, f'Could not read image:\\n{exc}', transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Image format and resolution audit (random sample; no image modification)

In [ ]:
sample = splits['train'].sample(n=min(500, len(splits['train'])), random_state=42)
rows=[]
for rel in sample.relative_path:
    p=DATASET_ROOT / Path(str(rel).replace('\\', '/'))
    try:
        with Image.open(p) as im:
            rows.append({'format': im.format, 'width':im.width, 'height':im.height})
    except Exception:
        rows.append({'format':'UNREADABLE', 'width':None, 'height':None})
image_stats=pd.DataFrame(rows)
print(image_stats['format'].value_counts(dropna=False))
print(image_stats[['width','height']].describe().round(1))
image_stats[['width','height']].hist(figsize=(9,3), bins=30)
plt.tight_layout();plt.show()

## 8. Export split metadata outside reports (safe copy, never delete originals)

In [ ]:
# Reproducibility: keep the exact pre-existing splits and class order.
NEW_SPLITS.mkdir(parents=True, exist_ok=True)
for name in ('train.csv', 'val.csv', 'test.csv', 'class_to_index.json'):
    source = SPLITS_DIR / name
    target = NEW_SPLITS / name
    if source.resolve() != target.resolve():
        if target.exists() and target.read_bytes() != source.read_bytes():
            raise RuntimeError(f'Existing target differs, refusing overwrite: {target}')
        if not target.exists():
            shutil.copy2(source, target)
    print(name, '->', target, 'OK:', source.read_bytes() == target.read_bytes())
print('Now update notebook paths and backend class mapping before considering any cleanup.')

## 9. Safe cleanup checklist

- Keep **original input images** at `C:\deep_learning_projectsrchive\images`.
- Keep the original five `ml/src` cleaning/splitting/verification scripts: this EDA notebook **does not replace their cleaning logic**.
- The exported `ml/data/splits/` CSV/JSON files are **required reproducibility assets**, not disposable reports.
- Update **every** notebook / script referring to `ml/data/splits`, plus `backend/app/services/inference.py`, to use `ml/data/splits` before removing the old folder.
- Existing metric JSONs and learning curves in `ml/reports/` may contain experiment evidence; inspect them before deletion.
- Commit the small CSV/JSON split metadata if repository policy permits; never commit raw image dataset or large `.h5` weights by default.